In [1]:
import re


class Expr:
    def __init__(self, op, args=None):
        self.op = op
        self.args = args or []

    def __repr__(self):
        if not self.args:
            return self.op
        if self.op in ('&', '|', '->', '<->'):
            return f"({self.args[0]} {self.op} {self.args[1]})"
        if self.op == '~':
            return f"~{self.args[0]}"
        if self.op in ('forall', 'exists'):
            return f"{self.op} {self.args[0]}.{self.args[1]}"
        return f"{self.op}({', '.join(map(str, self.args))})"

    def __eq__(self, other):
        return isinstance(other, Expr) and repr(self) == repr(other)

    def __hash__(self):
        return hash(repr(self))


def tokenize(s):
    return re.findall(r'forall|exists|<->|->|[().,~&|]|[A-Za-z0-9_]+', s)


def parse_fol(s):
    tokens = tokenize(s)

    def parse_iff():
        lhs = parse_imp()
        while tokens and tokens[0] == '<->':
            tokens.pop(0)
            lhs = Expr('<->', [lhs, parse_imp()])
        return lhs

    def parse_imp():
        lhs = parse_or()
        while tokens and tokens[0] == '->':
            tokens.pop(0)
            lhs = Expr('->', [lhs, parse_or()])
        return lhs

    def parse_or():
        lhs = parse_and()
        while tokens and tokens[0] == '|':
            tokens.pop(0)
            lhs = Expr('|', [lhs, parse_and()])
        return lhs

    def parse_and():
        lhs = parse_unary()
        while tokens and tokens[0] == '&':
            tokens.pop(0)
            lhs = Expr('&', [lhs, parse_unary()])
        return lhs

    def parse_unary():
        if tokens[0] == '~':
            tokens.pop(0)
            return Expr('~', [parse_unary()])
        if tokens[0] in ('forall', 'exists'):
            q = tokens.pop(0)
            var = Expr(tokens.pop(0))
            if tokens and tokens[0] == '.':
                tokens.pop(0)
            return Expr(q, [var, parse_unary()])
        if tokens[0] == '(':
            tokens.pop(0)
            e = parse_iff()
            if tokens and tokens[0] == ')':
                tokens.pop(0)
            return e
        name = tokens.pop(0)
        if tokens and tokens[0] == '(':
            tokens.pop(0)
            args = []
            while tokens and tokens[0] != ')':
                args.append(parse_unary())
                if tokens and tokens[0] == ',':
                    tokens.pop(0)
            if tokens and tokens[0] == ')':
                tokens.pop(0)
            return Expr(name, args)
        return Expr(name)

    return parse_iff()


# --- 2. The 6 Steps of CNF Conversion ---
def step1_eliminate_implications(e):
    if e.op == '<->':
        a, b = step1_eliminate_implications(e.args[0]), step1_eliminate_implications(e.args[1])
        return Expr('&', [Expr('|', [Expr('~', [a]), b]), Expr('|', [Expr('~', [b]), a])])
    if e.op == '->':
        a, b = step1_eliminate_implications(e.args[0]), step1_eliminate_implications(e.args[1])
        return Expr('|', [Expr('~', [a]), b])
    return Expr(e.op, [step1_eliminate_implications(a) for a in e.args])


def step2_move_negation_inward(e):
    if e.op == '~':
        sub = e.args[0]
        if sub.op == '~':
            return step2_move_negation_inward(sub.args[0])
        if sub.op == '&':
            return Expr('|', [step2_move_negation_inward(Expr('~', [a])) for a in sub.args])
        if sub.op == '|':
            return Expr('&', [step2_move_negation_inward(Expr('~', [a])) for a in sub.args])
        if sub.op == 'forall':
            return Expr('exists', [sub.args[0], step2_move_negation_inward(Expr('~', [sub.args[1]]))])
        if sub.op == 'exists':
            return Expr('forall', [sub.args[0], step2_move_negation_inward(Expr('~', [sub.args[1]]))])
        return Expr('~', [step2_move_negation_inward(sub)])
    return Expr(e.op, [step2_move_negation_inward(a) for a in e.args])


var_counter = 0


def step3_standardize_variables(e, env=None):
    global var_counter
    env = dict(env or {})
    if e.op in ('forall', 'exists'):
        var_counter += 1
        new_var = Expr(f"{e.args[0].op}_{var_counter}")
        env[e.args[0].op] = new_var
        return Expr(e.op, [new_var, step3_standardize_variables(e.args[1], env)])
    if not e.args and e.op in env:
        return env[e.op]
    return Expr(e.op, [step3_standardize_variables(a, env) for a in e.args])


skolem_counter = 0


def step4_skolemize(e, univ_vars=None):
    global skolem_counter
    univ_vars = list(univ_vars or [])
    if e.op == 'forall':
        return Expr('forall', [e.args[0], step4_skolemize(e.args[1], univ_vars + [e.args[0]])])
    if e.op == 'exists':
        skolem_counter += 1
        skolem_term = Expr(f"Sk{skolem_counter}", univ_vars) if univ_vars else Expr(f"SkConst{skolem_counter}")
        return step4_skolemize(substitute(e.args[1], {e.args[0].op: skolem_term}), univ_vars)
    return Expr(e.op, [step4_skolemize(a, univ_vars) for a in e.args])


def step5_drop_universal(e):
    if e.op == 'forall':
        return step5_drop_universal(e.args[1])
    return Expr(e.op, [step5_drop_universal(a) for a in e.args])


def step6_distribute_and_cnf(e):
    e = Expr(e.op, [step6_distribute_and_cnf(a) for a in e.args])
    if e.op == '|':
        a, b = e.args
        if a.op == '&':
            return step6_distribute_and_cnf(Expr('&', [Expr('|', [a.args[0], b]), Expr('|', [a.args[1], b])]))
        if b.op == '&':
            return step6_distribute_and_cnf(Expr('&', [Expr('|', [a, b.args[0]]), Expr('|', [a, b.args[1]])]))
    return e


def to_clauses(e):
    if e.op == '&':
        return to_clauses(e.args[0]) + to_clauses(e.args[1])

    def get_disjuncts(d):
        if d.op == '|':
            return get_disjuncts(d.args[0]) + get_disjuncts(d.args[1])
        return [d]

    return [frozenset(get_disjuncts(e))]


def fol_to_cnf(formula_str):
    e = parse_fol(formula_str)
    e = step1_eliminate_implications(e)
    e = step2_move_negation_inward(e)
    e = step3_standardize_variables(e)
    e = step4_skolemize(e)
    e = step5_drop_universal(e)
    e = step6_distribute_and_cnf(e)
    return to_clauses(e)


# --- 3. Unification & Resolution Engine with Graph Tracking ---
def is_var(x):
    return isinstance(x, Expr) and not x.args and x.op[0].islower()


def substitute(e, theta):
    if is_var(e) and e.op in theta:
        return substitute(theta[e.op], theta)
    return Expr(e.op, [substitute(a, theta) for a in e.args])


def occurs_check(v, x, theta):
    if v == x:
        return True
    if is_var(x) and x.op in theta:
        return occurs_check(v, theta[x.op], theta)
    return any(occurs_check(v, a, theta) for a in x.args)


def unify(x, y, theta=None):
    if theta is None:
        return None
    if x == y:
        return theta
    if is_var(x):
        if x.op in theta:
            return unify(theta[x.op], y, theta)
        if occurs_check(x, y, theta):
            return None
        return {**theta, x.op: y}
    if is_var(y):
        if y.op in theta:
            return unify(x, theta[y.op], theta)
        if occurs_check(y, x, theta):
            return None
        return {**theta, y.op: x}
    if x.op != y.op or len(x.args) != len(y.args):
        return None
    for a, b in zip(x.args, y.args):
        theta = unify(a, b, theta)
        if theta is None:
            return None
    return theta


rename_counter = 0


def freshen_clause(clause):
    """Rename variables in a clause so they don't collide during resolution."""
    global rename_counter
    rename_counter += 1
    suffix = f"_r{rename_counter}"
    mapping = {}

    def collect_vars(expr):
        if is_var(expr) and expr.op not in mapping:
            mapping[expr.op] = Expr(f"{expr.op}{suffix}")
        for a in expr.args:
            collect_vars(a)

    for lit in clause:
        collect_vars(lit)

    if not mapping:
        return frozenset(clause)
    return frozenset(substitute(lit, mapping) for lit in clause)


def format_clause(clause):
    if not clause:
        return "[]"
    return "{" + ", ".join(str(lit) for lit in sorted(clause, key=str)) + "}"


def format_theta(theta):
    if not theta:
        return "{}"
    return "{" + ", ".join(f"{k} -> {v}" for k, v in theta.items()) + "}"


class ClauseNode:
    def __init__(self, cid, literals, source, parents=None, theta=None):
        self.cid = cid
        self.literals = frozenset(literals)
        self.source = source        # e.g., "KB Premise 1", "Negated Query", or "Derived"
        self.parents = parents or []  # [parent_node_1, parent_node_2]
        self.theta = theta or {}

    def __str__(self):
        return f"[{self.cid}] {format_clause(self.literals)}"


def resolve_nodes(n1, n2, next_id_fn):
    c1 = freshen_clause(n1.literals)
    c2 = freshen_clause(n2.literals)
    resolvents = []

    for l1 in c1:
        for l2 in c2:
            theta = None
            if l1.op == '~' and l2.op != '~':
                theta = unify(l1.args[0], l2, {})
            elif l2.op == '~' and l1.op != '~':
                theta = unify(l1, l2.args[0], {})

            if theta is not None:
                rem = (c1 - {l1}) | (c2 - {l2})
                new_lits = frozenset(substitute(lit, theta) for lit in rem)
                node = ClauseNode(
                    cid=next_id_fn(),
                    literals=new_lits,
                    source="Derived",
                    parents=[n1, n2],
                    theta=theta,
                )
                resolvents.append(node)
    return resolvents


def fol_resolution_with_graph(kb_list, query_str, max_steps=100):
    counter = 0

    def next_id():
        nonlocal counter
        counter += 1
        return f"C{counter}"

    nodes_by_lits = {}
    all_nodes = []

    def add_initial_clause(clause, source):
        fclause = frozenset(clause)
        if fclause not in nodes_by_lits:
            node = ClauseNode(next_id(), fclause, source)
            nodes_by_lits[fclause] = node
            all_nodes.append(node)

    for idx, stmt in enumerate(kb_list, start=1):
        for c in fol_to_cnf(stmt):
            add_initial_clause(c, f"KB Premise {idx}")

    for c in fol_to_cnf(f"~({query_str})"):
        add_initial_clause(c, "Negated Query")

    # Check if empty clause is already present
    empty_lits = frozenset()
    if empty_lits in nodes_by_lits:
        return True, nodes_by_lits[empty_lits], all_nodes

    active_nodes = list(all_nodes)
    for _ in range(max_steps):
        new_nodes = []
        n_active = len(active_nodes)
        for i in range(n_active):
            for j in range(i + 1, n_active):
                for child in resolve_nodes(active_nodes[i], active_nodes[j], next_id):
                    if child.literals == empty_lits:
                        all_nodes.append(child)
                        return True, child, all_nodes
                    if child.literals not in nodes_by_lits:
                        nodes_by_lits[child.literals] = child
                        new_nodes.append(child)
                        all_nodes.append(child)

        if not new_nodes:
            return False, None, all_nodes
        active_nodes.extend(new_nodes)

    return False, None, all_nodes


# --- 4. Resolution Graph Display Helpers ---
def collect_proof_subgraph(goal_node):
    """Collect only nodes participating in the proof of goal_node."""
    needed = {}

    def dfs(node):
        if node.cid in needed:
            return
        needed[node.cid] = node
        for p in node.parents:
            dfs(p)

    dfs(goal_node)
    # Return in creation order (C1, C2, ...)
    return sorted(needed.values(), key=lambda n: int(n.cid[1:]))


def print_resolution_tree(node, prefix="", is_last=True, visited_in_branch=None):
    """Print an ASCII resolution tree from the empty clause down to base clauses."""
    if visited_in_branch is None:
        visited_in_branch = set()

    connector = "└── " if is_last else "├── "
    theta_str = f"  [θ: {format_theta(node.theta)}]" if node.parents else f"  ({node.source})"
    print(f"{prefix}{connector}{node}{theta_str}")

    if node.cid in visited_in_branch:
        print(f"{prefix}{'    ' if is_last else '│   '}└── (cycle back to {node.cid})")
        return

    next_visited = visited_in_branch | {node.cid}
    child_prefix = prefix + ("    " if is_last else "│   ")
    for idx, parent in enumerate(node.parents):
        print_resolution_tree(parent, child_prefix, idx == len(node.parents) - 1, next_visited)


def print_resolution_graph(goal_node):
    proof_nodes = collect_proof_subgraph(goal_node)

    print("\n=== RESOLUTION PROOF STEPS (NODES & EDGES) ===")
    for n in proof_nodes:
        if n.parents:
            p_ids = " + ".join(p.cid for p in n.parents)
            print(f"  {n.cid}: {format_clause(n.literals):<38} <- Resolve({p_ids}) with θ = {format_theta(n.theta)}")
        else:
            print(f"  {n.cid}: {format_clause(n.literals):<38} <- {n.source}")

    print("\n=== RESOLUTION GRAPH / TREE (TOP-DOWN FROM EMPTY CLAUSE) ===")
    print_resolution_tree(goal_node)


def export_dot(goal_node, filename="resolution_graph.dot"):
    """Export the proof graph in Graphviz DOT format."""
    proof_nodes = collect_proof_subgraph(goal_node)
    lines = ["digraph ResolutionGraph {", '  rankdir="BT";', '  node [shape=box, fontname="Consolas"];']
    for n in proof_nodes:
        label = f"{n.cid}: {format_clause(n.literals)}"
        color = "#ffcccc" if not n.literals else ("#e6f2ff" if not n.parents else "#f9f9f9")
        lines.append(f'  {n.cid} [label="{label}", style=filled, fillcolor="{color}"];')
    for n in proof_nodes:
        for p in n.parents:
            edge_label = format_theta(n.theta)
            lines.append(f'  {p.cid} -> {n.cid} [label=" {edge_label}"];')
    lines.append("}")
    with open(filename, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    return filename


# --- 5. User Input & Output ---
if __name__ == "__main__":
    print("Syntax: forall x. P(x), exists x. P(x), ~ (NOT), & (AND), | (OR), -> (IMP), <-> (IFF)")
    print("Rule: Variables start with lowercase (x, y); Constants/Predicates start with uppercase (John, Human).\n")

    n = int(input("Number of KB premises: "))
    kb = [input(f"Premise {i+1}: ") for i in range(n)]
    query = input("Query to prove: ")

    can_prove, empty_node, _ = fol_resolution_with_graph(kb, query)
    print("\n--- RESULT ---")
    print(f"Resolution Tree Formed: {'Yes (Empty clause [] derived)' if can_prove else 'No'}")
    print(f"Query Proven: {'TRUE' if can_prove else 'FALSE'}")

    if can_prove and empty_node is not None:
        print_resolution_graph(empty_node)
        dot_path = export_dot(empty_node)
        print(f"\nGraphviz DOT file saved to: {dot_path} (render with: dot -Tpng {dot_path} -o graph.png)")

Syntax: forall x. P(x), exists x. P(x), ~ (NOT), & (AND), | (OR), -> (IMP), <-> (IFF)
Rule: Variables start with lowercase (x, y); Constants/Predicates start with uppercase (John, Human).

Number of KB premises: 7
Premise 1: forall x. (Food(x) -> Likes(John, x))
Premise 2: Food(Apple) & Food(Vegetables)
Premise 3: forall x. forall y. ((Eats(x, y) & ~Killed(x)) -> Food(y))
Premise 4: Eats(Anil, Peanuts) & Alive(Anil)
Premise 5: forall x. (Eats(Anil, x) -> Eats(Harry, x))
Premise 6: forall x. (~Killed(x) -> Alive(x))
Premise 7: forall x. (Alive(x) -> ~Killed(x))
Query to prove: Likes(John, Peanuts)

--- RESULT ---
Resolution Tree Formed: Yes (Empty clause [] derived)
Query Proven: TRUE

=== RESOLUTION PROOF STEPS (NODES & EDGES) ===
  C1: {Likes(John, x_1), ~Food(x_1)}         <- KB Premise 1
  C4: {Food(y_3), Killed(x_2), ~Eats(x_2, y_3)} <- KB Premise 3
  C5: {Eats(Anil, Peanuts)}                  <- KB Premise 4
  C6: {Alive(Anil)}                          <- KB Premise 4
  C9: {~Aliv